# Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Dataset Load

In [ ]:
wustl_df = pd.read_csv("datasets/WUSTL/WUSTL_DATASET.csv")
train_df = pd.read_csv("datasets/CICIOMT/train/ciciomt2024_train_balanced.csv")
val_df = pd.read_csv("datasets/CICIOMT/val/ciciomt2024_validation_balanced.csv")
test_df = pd.read_csv("datasets/CICIOMT/test/ciciomt2024_test_balanced.csv")

# Feature Harmonization

In [ ]:
def harmonize_ciciomt(df):
    new_df = pd.DataFrame()

    new_df["duration"] = df["Duration"]
    new_df["rate"] = df["Rate"]

    # packet / byte volume
    new_df["total_bytes"] = df["Tot size"]
    new_df["total_packets"] = df["Number"]

    # packet size statistics
    new_df["avg_packet_size"] = df["AVG"]
    new_df["max_packet_size"] = df["Max"]
    new_df["min_packet_size"] = df["Min"]

    # timing / load
    new_df["inter_packet_time"] = df["IAT"]
    new_df["src_load"] = df["Srate"]
    new_df["dst_load"] = df["Drate"]

    # label
    new_df["label"] = df["binary_label"]

    return new_df

def harmonize_wustl(df):
    new_df = pd.DataFrame()

    new_df["duration"] = df["Dur"]
    new_df["rate"] = df["Rate"]

    # packet / byte volume
    new_df["total_bytes"] = df["TotBytes"]
    new_df["total_packets"] = df["TotPkts"]

    # avoid divide-by-zero
    new_df["avg_packet_size"] = df["TotBytes"] / df["TotPkts"].replace(0, np.nan)

    # packet size statistics
    new_df["max_packet_size"] = df[["sMaxPktSz", "dMaxPktSz"]].max(axis=1)
    new_df["min_packet_size"] = df[["sMinPktSz", "dMinPktSz"]].min(axis=1)

    # timing / load
    new_df["inter_packet_time"] = df[["SIntPkt", "DIntPkt"]].mean(axis=1)
    new_df["src_load"] = df["SrcLoad"]
    new_df["dst_load"] = df["DstLoad"]

    # label
    new_df["label"] = df["Label"]

    return new_df

ciciomt_train_h = harmonize_ciciomt(train_df)
ciciomt_val_h = harmonize_ciciomt(val_df)
ciciomt_test_h = harmonize_ciciomt(test_df)
wustl_h = harmonize_wustl(wustl_df)

In [ ]:
feature_cols_h = [
    "duration",
    "rate",
    "total_bytes",
    "total_packets",
    "avg_packet_size",
    "max_packet_size",
    "min_packet_size",
    "inter_packet_time",
    "src_load",
    "dst_load"
]

# Spearman correlation
ciciomt_corr = ciciomt_train_h[feature_cols_h].corr(method="spearman")
wustl_corr = wustl_h[feature_cols_h].corr(method="spearman")

print("CICIoMT Spearman correlation:")
print(ciciomt_corr)

print("\nWUSTL Spearman correlation:")
print(wustl_corr)

# CICIoMT2024 Harmonized Feature Correlation

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(ciciomt_corr, aspect="auto")
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(feature_cols_h)), feature_cols_h, rotation=90)
plt.yticks(range(len(feature_cols_h)), feature_cols_h)
plt.title("CICIoMT2024 Harmonized Feature Correlation")
plt.tight_layout()
plt.show()

# WUSTL-EHMS Harmonized Feature Correlation

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(wustl_corr, aspect="auto")
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(feature_cols_h)), feature_cols_h, rotation=90)
plt.yticks(range(len(feature_cols_h)), feature_cols_h)
plt.title("WUSTL-EHMS Harmonized Feature Correlation")
plt.tight_layout()
plt.show()

# Top Correlations

In [ ]:
def get_top_correlations(corr_matrix, dataset_name, top_n=10):
    pairs = []

    cols = corr_matrix.columns

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            pairs.append({
                "dataset": dataset_name,
                "feature_1": cols[i],
                "feature_2": cols[j],
                "spearman_corr": corr_matrix.iloc[i, j],
                "abs_corr": abs(corr_matrix.iloc[i, j])
            })

    pairs_df = pd.DataFrame(pairs)
    pairs_df = pairs_df.sort_values(by="abs_corr", ascending=False)

    return pairs_df.head(top_n)

top_ciciomt_corr = get_top_correlations(ciciomt_corr, "CICIoMT2024")
top_wustl_corr = get_top_correlations(wustl_corr, "WUSTL-EHMS")

print("Top CICIoMT correlations:")
print(top_ciciomt_corr)

print("\nTop WUSTL correlations:")
print(top_wustl_corr)